# 3) Multi-period dispatch — Exercises

**Goals:** check ramp feasibility and simulate a battery by hand, build your own 24-hour Linopy model with a flexible coal ramp and battery size and match PyPSA, put a price on flexibility, and find where extra storage stops paying.

The setup cell holds the lecture's day (`LOAD`, `FREE`, `PLANTS`, `ETA`) and a PyPSA helper for checking. Power in MW, energy in MWh (one-hour steps), prices in €/MWh, cost in € per day.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

# the lecture's day
T = 24
HOURS = np.arange(T)
LOAD = 1900 + 400 * np.sin(np.pi * (HOURS - 6) / 12).clip(0) + 250 * np.exp(-((HOURS - 19) / 2.0) ** 2)
FREE = 2200 * 0.8 * np.clip(np.sin(np.pi * (HOURS - 6) / 13), 0, None) + 400.0
PLANTS = pd.DataFrame({"p_nom": [900, 900, 500], "mc": [60, 95, 150], "ramp": [60, 300, 500]},
                      index=pd.Index(["coal", "ccgt", "ocgt"], name="gen"))
ETA = 0.95


def pypsa_day(coal_ramp=60.0, battery_mw=300.0, battery_mwh=1200.0):
    """The lecture's day in PyPSA, for checking. Returns (cost EUR, hourly prices EUR/MWh)."""
    n = pypsa.Network()
    n.set_snapshots(HOURS)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=LOAD)
    n.add("Generator", "free", bus="HR", p_nom=FREE.max(), p_max_pu=FREE / FREE.max(), marginal_cost=0)
    for gen, row in PLANTS.iterrows():
        ramp = coal_ramp if gen == "coal" else row.ramp
        n.add("Generator", gen, bus="HR", p_nom=row.p_nom, marginal_cost=row.mc,
              ramp_limit_up=ramp / row.p_nom, ramp_limit_down=ramp / row.p_nom)
    if battery_mwh > 0:
        n.add("StorageUnit", "battery", bus="HR", p_nom=battery_mw, max_hours=battery_mwh / battery_mw,
              efficiency_store=ETA, efficiency_dispatch=ETA, cyclic_state_of_charge=True)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n.objective, n.buses_t.marginal_price["HR"].to_numpy()

### Warm-ups

**1. Is a schedule ramp-feasible?**

Write `ramp_violation(profile, ramp)` returning the largest amount (MW) by which consecutive hours of a schedule exceed the ramp limit (0 if it is feasible).

In [2]:
def ramp_violation(profile, ramp: float) -> float:
    """Largest excess of |g[t] - g[t-1]| over the ramp limit (MW); 0 if feasible."""
    ### BEGIN SOLUTION
    steps = np.abs(np.diff(np.asarray(profile, float)))
    return float(max(steps.max() - ramp, 0.0)) if len(steps) else 0.0
    ### END SOLUTION

In [3]:
assert ramp_violation([900, 840, 780, 840], 60) == 0
assert np.isclose(ramp_violation([900, 900, 600], 60), 240)
assert ramp_violation([500], 10) == 0

**2. Simulate a battery**

Write `soc_path(charge, discharge, e_start, eta=ETA)` returning the state of charge after each hour (MWh): e[t] = e[t−1] + η·c[t] − d[t]/η.

In [4]:
def soc_path(charge, discharge, e_start: float, eta: float = ETA) -> np.ndarray:
    """State of charge (MWh) at the end of each hour."""
    ### BEGIN SOLUTION
    flows = eta * np.asarray(charge, float) - np.asarray(discharge, float) / eta
    return e_start + np.cumsum(flows)
    ### END SOLUTION

In [5]:
path = soc_path([100, 100, 0, 0], [0, 0, 90.25, 90.25], e_start=0)
assert np.allclose(path, [95, 190, 95, 0]), "charge 200 MWh, get back 0.95 x 0.95 x 200 = 180.5 MWh"
assert np.isclose(soc_path([0], [95], e_start=100)[-1], 0), "discharging 95 MWh empties 100 MWh of storage"

### Core

**3. Your own model, with knobs**

Write `dispatch(coal_ramp, battery_mw=300, battery_mwh=1200)` — the lecture's Linopy model with the coal ramp limit and the battery size as parameters (`battery_mwh=0` means no battery). Return a dict with `cost`, `prices` (array) and `curtailed` (MWh). Check it against `pypsa_day` for three different settings.

In [6]:
def dispatch(coal_ramp: float, battery_mw: float = 300.0, battery_mwh: float = 1200.0) -> dict:
    """24-hour dispatch with ramp limits and an optional cyclic battery."""
    ### BEGIN SOLUTION
    m = linopy.Model()
    t = pd.RangeIndex(T, name="t")
    g = m.add_variables(lower=0, upper=xr.DataArray(np.tile(PLANTS.p_nom.to_numpy(), (T, 1)), coords=[t, PLANTS.index]),
                        name="g")
    f = m.add_variables(lower=0, upper=xr.DataArray(FREE, coords=[t]), name="free")
    supply = g.sum("gen") + f
    if battery_mwh > 0:
        c = m.add_variables(lower=0, upper=battery_mw, coords=[t], name="charge")
        d = m.add_variables(lower=0, upper=battery_mw, coords=[t], name="discharge")
        e = m.add_variables(lower=0, upper=battery_mwh, coords=[t], name="soc")
        m.add_constraints(e - e.roll(t=1) - ETA * c + d / ETA == 0, name="soc_balance")
        supply = supply + d - c
    m.add_constraints(supply == xr.DataArray(LOAD, coords=[t]), name="balance")
    ramps = PLANTS.ramp.copy()
    ramps["coal"] = coal_ramp
    r = xr.DataArray(ramps.to_numpy(), coords=[PLANTS.index])
    later = xr.DataArray(HOURS > 0, coords=[t])
    m.add_constraints(g - g.shift(t=1) <= r, name="ramp_up", mask=later)
    m.add_constraints(g.shift(t=1) - g <= r, name="ramp_down", mask=later)
    m.add_objective((g * xr.DataArray(PLANTS.mc.to_numpy(), coords=[PLANTS.index])).sum())
    m.solve(solver_name="highs", output_flag=False)
    return {"cost": m.objective.value, "prices": m.constraints["balance"].dual.to_numpy() + 0.0,
            "curtailed": float((FREE - m.variables["free"].solution).sum())}
    ### END SOLUTION

In [7]:
for settings in [dict(coal_ramp=60), dict(coal_ramp=150, battery_mwh=0), dict(coal_ramp=30, battery_mw=200, battery_mwh=400)]:
    mine = dispatch(**settings)
    cost_ref, prices_ref = pypsa_day(**settings)
    assert np.isclose(mine["cost"], cost_ref, rtol=1e-9), settings
    assert np.abs(mine["prices"] - prices_ref).max() < 1e-6, settings
assert np.isclose(dispatch(60)["cost"], 1_837_742.65, atol=0.01), "the lecture's ramp + battery case"
print("own model = PyPSA (cost and every hourly price) for all three settings")

own model = PyPSA (cost and every hourly price) for all three settings


**4. What is a flexible coal plant worth?**

Without a battery, sweep the coal ramp limit over 20, 40, 60, 100, 150, 200 and 300 MW/h. Write `ramp_sweep(ramps)` returning a DataFrame (index: ramp) with daily cost, curtailment and maximum price. Find the smallest ramp in your sweep at which no solar is curtailed.

In [8]:
def ramp_sweep(ramps) -> pd.DataFrame:
    """Cost (EUR/day), curtailment (MWh) and maximum price (EUR/MWh) for each coal ramp limit, without battery."""
    ### BEGIN SOLUTION
    rows = {}
    for r in ramps:
        res = dispatch(r, battery_mwh=0)
        rows[r] = {"cost": res["cost"], "curtailed": res["curtailed"], "max_price": res["prices"].max()}
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis("coal_ramp_MW_per_h")
    ### END SOLUTION

In [9]:
sweep = ramp_sweep([20, 40, 60, 100, 150, 200, 300])
assert sweep.cost.is_monotonic_decreasing, "a more flexible plant never makes the day more expensive"
no_curtailment = sweep.index[sweep.curtailed < 1e-6].min()
assert sweep.loc[20, "curtailed"] > sweep.loc[60, "curtailed"] > 0
print(f"no curtailment from a ramp of {no_curtailment} MW/h; going from 20 to 300 MW/h saves "
      f"{sweep.loc[20, 'cost'] - sweep.loc[300, 'cost']:,.0f} EUR/day")
sweep.round(1)

no curtailment from a ramp of 300 MW/h; going from 20 to 300 MW/h saves 222,550 EUR/day


                         cost  curtailed  max_price
coal_ramp_MW_per_h                                 
20                  2056014.8     3269.9      150.0
40                  2001255.8     1948.7      150.0
60                  1955862.3     1322.7      150.0
100                 1891084.8      243.4      150.0
150                 1849034.1       19.5      150.0
200                 1838540.9       17.8      150.0
300                 1833465.1        0.0      150.0

**5. Interpretation**

Your sweep puts a price on flexibility. Why does the saving per extra MW/h of ramp rate shrink as the plant gets more flexible? And why do inflexible plants make solar curtailment — and zero or negative prices — more likely, even when total supply would be enough?

<!-- BEGIN SOLUTION -->
A slow plant cannot follow the daily shape of residual demand (demand minus solar): it must start ramping down hours before the solar peak and still stays above what the midday hours need, then cannot ramp back up fast enough for the evening. Each extra MW/h of ramp rate lets it track the shape more closely, which avoids curtailing free solar at midday and avoids starting expensive peakers in the evening. The first increments fix the worst mismatches and save the most; once the plant can follow the residual-demand curve within a few hours, further flexibility only shaves small remaining mismatches — in the sweep the saving per MW/h falls from about 2,700 €/day (20 → 40 MW/h) to about 50 €/day (200 → 300 MW/h). Once a plant can follow even the steepest hour-to-hour change in residual demand, more ramp rate is worth nothing: a concave value, as with line capacity in Maths-3. Curtailment and zero prices appear because the inflexible plant's minimum feasible output, set by its ramp path rather than by its technical minimum, plus the free energy, exceeds midday demand: the system has enough energy over the day but not the ability to move it in time. That is why flexibility — fast plants, storage, flexible demand, exports — is the counterpart of large solar shares.
<!-- END SOLUTION -->

### Challenge

**6. How much storage is worth building?**

With the 60 MW/h coal ramp and a 300 MW battery, sweep the battery's energy capacity over 0, 150, 300, 600, 1200, 1800, 2400 and 3600 MWh. Write `storage_sweep(sizes)` returning, for each size, the daily cost, the saving relative to no battery, and the **marginal** saving of the last step per MWh of added capacity. Confirm one size with PyPSA.

In [10]:
def storage_sweep(sizes) -> pd.DataFrame:
    """Daily cost, total saving vs no battery, and marginal saving per added MWh, for each battery energy size."""
    ### BEGIN SOLUTION
    costs = pd.Series({s: dispatch(60, battery_mw=300, battery_mwh=s)["cost"] for s in sizes})
    out = pd.DataFrame({"cost": costs, "saving": costs.iloc[0] - costs})
    out["marginal_saving_per_MWh"] = (-costs.diff() / pd.Series(sizes, index=costs.index).diff()) + 0.0
    return out.rename_axis("battery_MWh")
    ### END SOLUTION

In [11]:
storage = storage_sweep([0, 150, 300, 600, 1200, 1800, 2400, 3600])
marginal = storage.marginal_saving_per_MWh.dropna()
assert (storage.cost.diff().dropna() <= 1e-6).all(), "more storage never raises cost"
assert marginal.is_monotonic_decreasing, "each extra MWh of storage is worth less than the one before"
assert np.isclose(marginal.iloc[-1], 0, atol=1e-6), "beyond some size, extra storage energy is worth nothing"
cost_ref, _ = pypsa_day(60, 300, 600)
assert np.isclose(storage.loc[600, "cost"], cost_ref, rtol=1e-9)
storage.round(1)

                  cost    saving  marginal_saving_per_MWh
battery_MWh                                              
0            1955862.3       0.0                      NaN
150          1934220.7   21641.6                    144.3
300          1918967.7   36894.6                    101.7
600          1891892.7   63969.6                     90.2
1200         1837742.7  118119.6                     90.2
1800         1802595.2  153267.1                     58.6
2400         1800509.8  155352.5                      3.5
3600         1800509.8  155352.5                      0.0

**7. Interpretation**

Where does the value of extra storage energy run out in this day, and why? Explain the plateau at 90.25 €/MWh. A developer argues "the first 150 MWh earned X € per MWh per day, so a 2,400 MWh battery will earn 16 times as much". What is wrong with that, and what would you need to look at beyond this single day before deciding on a size?

<!-- BEGIN SOLUTION -->
The battery takes on jobs in order of value. The first MWh replace the 150 €/MWh OCGT in the evening (about 144 €/MWh of capacity). Next it stores midday solar that would otherwise be curtailed and delivers it at night instead of 95 €/MWh gas: one extra MWh of capacity holds one MWh of free solar and returns 0.95 MWh, so each MWh is worth 95 × 0.95 = 90.25 € — the plateau between 300 and 1,200 MWh. Once that surplus is used up, a bigger battery lets the slow coal plant ramp down less (its midday minimum rises from 379 to 519 MW) and stores its 60 €/MWh output for the night, which still beats gas after losses — the battery even accepts a little more solar curtailment to do so. That job is worth less (58.58 €/MWh, then 3.48 €/MWh), and somewhere between 2,400 and 3,600 MWh it runs out: with a 300 MW power rating and coal's ramp path, there is no profitable shift left, and extra energy capacity is worth exactly zero. Note what this says about storage: it serves whatever is cheapest to store, which is not always renewable energy. The developer's argument assumes a constant value per MWh, but storage value is strongly concave: the first MWh take the most valuable job (here, replacing 150 €/MWh OCGT output with free solar), later ones take less valuable jobs, and eventually none. Before sizing, you would look at a full year of days (sunny and cloudy, weekday and weekend, winter and summer), at how the marginal value of power (MW) and energy (MWh) differ, at the battery's own effect on prices (more batteries flatten the price spread they earn from, M2.2's cannibalisation), at degradation and cycle limits, and at the annualised cost per MWh of capacity — which is exactly what capacity-expansion models in Module 05 do.
<!-- END SOLUTION -->